## Catalog-Plugin mit Component-Tab

Das Plugin wird mit dem offiziellen CLI-Template erzeugt. 

Eine `EntityContentBlueprint`-Extension ergänzt Catalog-Entities um einen Tab.

Der Filter beschränkt den Tab auf `Component`-Entities.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn new --select frontend-plugin --option pluginId=openai-tab

Der Befehl installiert die benötigten Abhängigkeiten.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn --cwd plugins/openai-tab add   @backstage/frontend-plugin-api   @backstage/plugin-catalog-react

Inhalt des Tabs.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
rm -rf plugins/openai-tab/src/components/*

cat > plugins/openai-tab/src/components/OpenAITab.tsx <<'EOF'
import React, { useState } from 'react';
import { useEntity } from '@backstage/plugin-catalog-react';
import { Content, Header, Page } from '@backstage/core-components';
import {
  TextField,
  Button,
  Box,
  Typography,
  CircularProgress,
} from '@mui/material';
import {
  discoveryApiRef,
  fetchApiRef,
  useApi,
} from '@backstage/core-plugin-api';
import ReactMarkdown from 'react-markdown';

export const OpenAITab = () => {
  const { entity } = useEntity();

  const discoveryApi = useApi(discoveryApiRef);
  const fetchApi = useApi(fetchApiRef);

  const projectSlug =
    entity.metadata.annotations?.['gitlab.com/project-slug'];

  const slugName = projectSlug?.split('/').pop();
  const vectorStoreName = slugName || entity.metadata.name;

  const [input, setInput] = useState('');
  const [response, setResponse] = useState<string | null>(null);
  const [loading, setLoading] = useState(false);
  const [error, setError] = useState<string | null>(null);

  const handleSubmit = async () => {
    if (!input.trim() || loading) {
      return;
    }

    setLoading(true);
    setError(null);
    setResponse(null);

    try {
      const backendUrl =
        await discoveryApi.getBaseUrl('openai-tab');

      const res = await fetchApi.fetch(`${backendUrl}/query`, {
        method: 'POST',
        headers: {
          'Content-Type': 'application/json',
        },
        body: JSON.stringify({
          vectorStoreName,
          query: input.trim(),
        }),
      });

      const data = await res.json();

      if (!res.ok) {
        throw new Error(
          data.error || `Backend-Fehler ${res.status}`,
        );
      }

      setResponse(data.text || 'Keine Antwort erhalten.');
    } catch (e) {
      setError(
        e instanceof Error
          ? e.message
          : 'Unbekannter Verbindungsfehler',
      );
    } finally {
      setLoading(false);
    }
  };

  return (
    <Page themeId="tool">
      <Header title="OpenAI" />

      <Content>
        <Typography variant="h6">
          OpenAI für <strong>{entity.metadata.name}</strong>
        </Typography>

        <Typography variant="body2" color="textSecondary">
          Vector Store: {vectorStoreName}
        </Typography>

        <Box mt={4} mb={2}>
          {loading && <CircularProgress />}

          {!loading && response && (
            <Box
              border={1}
              borderColor="grey.300"
              borderRadius={2}
              p={2}
            >
              <ReactMarkdown>{response}</ReactMarkdown>
            </Box>
          )}

          {!loading && !response && (
            <Typography variant="body2" color="textSecondary">
              Noch keine Antwort.
            </Typography>
          )}
        </Box>

        <Box display="flex" gap={2}>
          <TextField
            fullWidth
            label="Frage an OpenAI"
            value={input}
            disabled={loading}
            onChange={event => setInput(event.target.value)}
            onKeyDown={event => {
              if (event.key === 'Enter') {
                event.preventDefault();
                void handleSubmit();
              }
            }}
          />

          <Button
            variant="contained"
            onClick={() => void handleSubmit()}
            disabled={loading || !input.trim()}
          >
            Senden
          </Button>
        </Box>

        {error && (
          <Box mt={2}>
            <Typography color="error">{error}</Typography>
          </Box>
        )}
      </Content>
    </Page>
  );
};
EOF


Catalog-Tab gemäss offizieller `EntityContentBlueprint`-Anleitung.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

cat > plugins/openai-tab/src/plugin.tsx <<'EOF'
import React from 'react';
import { createFrontendPlugin } from '@backstage/frontend-plugin-api';
import { EntityContentBlueprint } from '@backstage/plugin-catalog-react/alpha';

const openAIContent = EntityContentBlueprint.make({
  params: {
    path: 'openai',
    title: 'OpenAI',
    filter: 'kind:component',
    loader: () =>
      import('./components/OpenAITab').then(m => <m.OpenAITab />),
  },
});

export const openaiTabPlugin = createFrontendPlugin({
  pluginId: 'openai-tab',
  extensions: [openAIContent],
});
EOF

cat > plugins/openai-tab/src/index.ts <<'EOF'
export { openaiTabPlugin as default } from './plugin';
EOF


In [ ]:
%%bash
cd ~/mybackstage/
cat > app-config.openai.yaml <<'EOF'
catalog:
  locations:
    # TBZ IT GB
    - type: url
      target: https://gitlab.com/ch-tbz-it/Stud/allgemein/backstage-it/-/blob/main/catalog-info.yaml
      rules:
        - allow: [Domain, System, User, Group, Resource, Template]

    # TBZ IT HF
    - type: url
      target: https://gitlab.com/ch-tbz-wb/general/backstage-wb/-/blob/main/catalog-info.yaml
      rules:
        - allow: [Domain, System, User, Group, Resource, Template]
EOF


**Backstage starten:** 

In Backstage Component `cna` suchen und auf Tab "OpenAI" wechseln.

Eine Frage stellen wie:
* was wird in diesem Modul unterrichtet?

In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage OpenAI"
export BACKSTAGE_PORT="3001"

echo "http://$(cat ~/data/server-ip):${BACKSTAGE_PORT}"
source ~/.nvm/nvm.sh
cd ~/mybackstage
yarn start  --config ~/mybackstage/app-config.yaml \
            --config ~/mybackstage/app-config.test.yaml \
            --config ~/mybackstage/app-config.openai.yaml
